# Strong Tabular Baseline

This notebook trains several tabular models, validates with the official RMSE metric, and writes a Kaggle-ready `submission.csv`.

It supports two layouts:

- Standard Kaggle tabular layout: `train.csv`, `test.csv`, `sample_submission.csv`.
- This wellbore competition layout: `train/`, `test/`, `sample_submission.csv`.

In [ ]:
# ============================================================
# 0. Configuration
# ============================================================

from pathlib import Path
import importlib
import subprocess
import sys
import warnings

import joblib
import numpy as np
import pandas as pd
from tqdm.auto import tqdm

from sklearn.base import clone
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import ExtraTreesRegressor, RandomForestRegressor
from sklearn.impute import SimpleImputer
from sklearn.linear_model import Ridge
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import GroupKFold, KFold
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler

warnings.filterwarnings("ignore")
pd.set_option("display.max_columns", 120)

# Leave this as None when the notebook is inside the data folder or running on Kaggle.
# Local Windows example:
# DATA_ROOT_OVERRIDE = Path(r"C:\Users\alem\Documents\codes\kaggle_rogii")
# Kaggle example:
# DATA_ROOT_OVERRIDE = Path("/kaggle/input/your-dataset-folder")
DATA_ROOT_OVERRIDE = None

RANDOM_STATE = 42
N_SPLITS = 5
ENSEMBLE_TOP_N = 3

# Keep False for Kaggle code competitions because internet must be disabled.
# Set True only in a local environment if you want the notebook to try pip installs.
ALLOW_PIP_INSTALL = False

# If runtime is too long, set this to a number like 500_000.
# None means use all valid training rows.
MAX_TRAIN_ROWS = None

OUTPUT_ROOT = Path("/kaggle/working") if Path("/kaggle/working").exists() else Path.cwd()
MODEL_DIR = OUTPUT_ROOT / "models"
SUBMISSION_DIR = OUTPUT_ROOT / "submissions"
MODEL_DIR.mkdir(parents=True, exist_ok=True)
SUBMISSION_DIR.mkdir(parents=True, exist_ok=True)


## 1. Optional Model Imports

Ridge and ExtraTrees always run through scikit-learn. LightGBM, XGBoost, and CatBoost run if they are available in the environment. Kaggle code competitions require internet off, so this notebook does not rely on runtime installs.

In [ ]:
# ============================================================
# 1. Optional imports
# ============================================================

def optional_import(module_name, pip_name=None):
    """Import an optional package, optionally trying pip install in local runs."""
    pip_name = pip_name or module_name
    try:
        return importlib.import_module(module_name)
    except ImportError:
        if ALLOW_PIP_INSTALL:
            print(f"Installing {pip_name}...")
            subprocess.check_call([sys.executable, "-m", "pip", "install", pip_name])
            return importlib.import_module(module_name)
        print(f"Optional package not available, skipping {module_name}: install {pip_name} locally if needed.")
        return None


lgb = optional_import("lightgbm")
xgb = optional_import("xgboost")
catboost = optional_import("catboost")


## 2. Find Data and Load Files

The loader first looks for `train.csv` and `test.csv`. If they are not present, it loads the per-well CSV files from `train/` and `test/`.

In [ ]:
# ============================================================
# 2. Data discovery
# ============================================================

def is_standard_root(path: Path) -> bool:
    return (path / "train.csv").exists() and (path / "test.csv").exists() and (path / "sample_submission.csv").exists()


def is_wellbore_root(path: Path) -> bool:
    return (path / "train").exists() and (path / "test").exists() and (path / "sample_submission.csv").exists()


def describe_available_inputs() -> str:
    lines = [f"Current working directory: {Path.cwd()}"]
    kaggle_input = Path("/kaggle/input")
    if kaggle_input.exists():
        lines.append("/kaggle/input contents:")
        for path in sorted(kaggle_input.iterdir()):
            if path.is_dir():
                child_names = ", ".join(sorted(p.name for p in path.iterdir())[:10])
                lines.append(f"  - {path} -> {child_names}")
            else:
                lines.append(f"  - {path}")
    return "\n".join(lines)


def candidate_roots():
    roots = []
    if DATA_ROOT_OVERRIDE is not None:
        roots.append(Path(DATA_ROOT_OVERRIDE))
    roots.extend([Path.cwd(), Path.cwd().parent])
    if Path(r"C:\Users\alem\Documents\codes\kaggle_rogii").exists():
        roots.append(Path(r"C:\Users\alem\Documents\codes\kaggle_rogii"))
    if Path("/kaggle/input").exists():
        roots.append(Path("/kaggle/input"))
        roots.extend([p for p in Path("/kaggle/input").iterdir() if p.is_dir()])

    seen = set()
    for root in roots:
        key = str(root)
        if key not in seen:
            seen.add(key)
            yield root


def find_data_root() -> Path:
    checked = []
    for root in candidate_roots():
        checked.append(str(root))
        if root.exists() and (is_standard_root(root) or is_wellbore_root(root)):
            return root

        # Search one level deeper only. Avoid broad recursive scans into system folders.
        if root.exists() and root.name == "input":
            for child in root.iterdir():
                checked.append(str(child))
                if child.is_dir() and (is_standard_root(child) or is_wellbore_root(child)):
                    return child

    raise FileNotFoundError(
        "Could not find train.csv/test.csv or train//test/ plus sample_submission.csv.\n"
        + describe_available_inputs()
        + "\n\nChecked:\n"
        + "\n".join(checked)
    )


DATA_ROOT = find_data_root()
SAMPLE_SUBMISSION_PATH = DATA_ROOT / "sample_submission.csv"
sample_submission = pd.read_csv(SAMPLE_SUBMISSION_PATH)

print("Data root:", DATA_ROOT)
print("Output root:", OUTPUT_ROOT)
print("Sample submission shape:", sample_submission.shape)
display(sample_submission.head())


In [ ]:
# ============================================================
# 3. Feature engineering and data loading
# ============================================================

def stable_text_code(value) -> int:
    """Turn a text label into a stable small numeric code without Python's randomized hash."""
    if pd.isna(value):
        return -1
    return sum(ord(ch) for ch in str(value))


def summarize_typewell(typewell_path: Path) -> dict:
    """Create simple summary features from the vertical reference log."""
    typewell = pd.read_csv(typewell_path)
    geology = typewell.get("Geology", pd.Series(index=typewell.index, dtype="object"))
    geology_mode = geology.dropna().mode()
    mode_value = geology_mode.iloc[0] if len(geology_mode) else np.nan

    return {
        "typewell_rows": len(typewell),
        "typewell_tvt_min": typewell["TVT"].min(),
        "typewell_tvt_max": typewell["TVT"].max(),
        "typewell_tvt_range": typewell["TVT"].max() - typewell["TVT"].min(),
        "typewell_gr_mean": typewell["GR"].mean(),
        "typewell_gr_std": typewell["GR"].std(),
        "typewell_gr_min": typewell["GR"].min(),
        "typewell_gr_max": typewell["GR"].max(),
        "typewell_geology_unique": geology.nunique(dropna=True),
        "typewell_geology_mode_code": stable_text_code(mode_value),
    }


def add_well_features(horizontal: pd.DataFrame, well: str, typewell_features: dict) -> pd.DataFrame:
    """Add safe row-level features available in both train and test."""
    df = horizontal.copy()
    df["well"] = well
    df["row_index"] = np.arange(len(df))
    df["row_fraction"] = df["row_index"] / max(len(df) - 1, 1)
    df["id"] = well + "_" + df["row_index"].astype(str)

    for col in ["MD", "X", "Y", "Z"]:
        if col in df.columns:
            lower = col.lower()
            df[f"{lower}_rel"] = df[col] - df[col].iloc[0]
            df[f"{lower}_diff_1"] = df[col].diff().fillna(0)

    if "GR" in df.columns:
        df["gr_diff_1"] = df["GR"].diff().fillna(0)
        df["gr_roll_mean_5"] = df["GR"].rolling(5, min_periods=1, center=True).mean()
        df["gr_roll_mean_25"] = df["GR"].rolling(25, min_periods=1, center=True).mean()
        df["gr_roll_std_25"] = df["GR"].rolling(25, min_periods=2, center=True).std()

    # TVT_input is a target copy outside the evaluation zone, so do not use it directly.
    # Derived anchor features are allowed because they use only the provided input column.
    if "TVT_input" in df.columns:
        known = df["TVT_input"].notna()
        idx = pd.Series(np.arange(len(df)), index=df.index)
        prev_known_idx = idx.where(known).ffill()
        next_known_idx = idx.where(known).bfill()
        df["tvt_input_is_missing"] = df["TVT_input"].isna().astype(int)
        df["tvt_input_ffill"] = df["TVT_input"].ffill()
        df["tvt_input_bfill"] = df["TVT_input"].bfill()
        df["tvt_input_linear"] = df["TVT_input"].interpolate(method="linear", limit_direction="both")
        df["dist_prev_tvt_input"] = idx - prev_known_idx
        df["dist_next_tvt_input"] = next_known_idx - idx

    for name, value in typewell_features.items():
        df[name] = value

    return df


def load_well_folder(folder: Path, has_target: bool) -> pd.DataFrame:
    frames = []
    files = sorted(folder.glob("*__horizontal_well.csv"))
    for horizontal_path in tqdm(files, desc=f"Loading {folder.name} wells"):
        well = horizontal_path.name.split("__")[0]
        typewell_path = folder / f"{well}__typewell.csv"
        horizontal = pd.read_csv(horizontal_path)
        typewell_features = summarize_typewell(typewell_path)
        frame = add_well_features(horizontal, well, typewell_features)
        if has_target and "TVT" not in frame.columns:
            raise ValueError(f"Missing TVT target in {horizontal_path}")
        frames.append(frame)
    return pd.concat(frames, ignore_index=True)


def infer_target_column(train_df: pd.DataFrame, test_df: pd.DataFrame, sample_df: pd.DataFrame) -> str:
    """Infer target from sample submission, falling back to train-only columns."""
    sample_target = sample_df.columns[-1]
    if sample_target in train_df.columns:
        return sample_target
    for col in train_df.columns:
        if col.lower() == sample_target.lower():
            return col

    train_only = [col for col in train_df.columns if col not in test_df.columns]
    numeric_train_only = [col for col in train_only if pd.api.types.is_numeric_dtype(train_df[col])]
    if len(numeric_train_only) == 1:
        return numeric_train_only[0]
    if "TVT" in train_df.columns:
        return "TVT"
    raise ValueError(f"Could not infer target column. Train-only columns: {train_only}")


if is_standard_root(DATA_ROOT):
    DATA_MODE = "flat_csv"
    train_df = pd.read_csv(DATA_ROOT / "train.csv")
    test_df = pd.read_csv(DATA_ROOT / "test.csv")
    GROUP_COL = next((c for c in ["well", "well_id", "WELL", "WELLNAME", "wellname"] if c in train_df.columns), None)
else:
    DATA_MODE = "well_folder"
    train_df = load_well_folder(DATA_ROOT / "train", has_target=True)
    test_df = load_well_folder(DATA_ROOT / "test", has_target=False)
    GROUP_COL = "well"

TARGET_COL = infer_target_column(train_df, test_df, sample_submission)
SUBMISSION_ID_COL = sample_submission.columns[0]
SUBMISSION_TARGET_COL = sample_submission.columns[-1]

print("Data mode:", DATA_MODE)
print("Train shape:", train_df.shape)
print("Test shape:", test_df.shape)
print("Target column:", TARGET_COL)
print("Submission columns:", list(sample_submission.columns))
print("Group column:", GROUP_COL)
display(train_df.head())


## 3. Leakage-Safe Feature Selection

For the wellbore layout, validation and training use rows where `TVT_input` is missing. Those rows match the hidden evaluation zone. The raw `TVT_input` column is excluded because it is a target copy outside the evaluation zone.

In [ ]:
# ============================================================
# 4. Leakage-safe modeling table
# ============================================================

def build_training_mask(df: pd.DataFrame) -> pd.Series:
    mask = df[TARGET_COL].notna()
    if DATA_MODE == "well_folder" and "TVT_input" in df.columns:
        eval_like_mask = df["TVT_input"].isna()
        if eval_like_mask.sum() > 0:
            mask &= eval_like_mask
        else:
            print("Warning: no missing TVT_input rows found in train; using all labeled rows.")
    return mask


def choose_features(train_df: pd.DataFrame, test_df: pd.DataFrame) -> list:
    common_cols = [col for col in train_df.columns if col in test_df.columns]
    leak_or_id_cols = {
        TARGET_COL,
        TARGET_COL.lower(),
        TARGET_COL.upper(),
        SUBMISSION_ID_COL,
        "id",
        "well",
        "WELL",
        "WELLNAME",
        "wellname",
        "TVT",
        "tvt",
        "TVT_input",
    }
    if GROUP_COL is not None:
        leak_or_id_cols.add(GROUP_COL)
    features = [col for col in common_cols if col not in leak_or_id_cols]
    return features


train_mask = build_training_mask(train_df)
model_df = train_df.loc[train_mask].copy().reset_index(drop=True)

if MAX_TRAIN_ROWS is not None and len(model_df) > MAX_TRAIN_ROWS:
    model_df = model_df.sample(MAX_TRAIN_ROWS, random_state=RANDOM_STATE).reset_index(drop=True)
    print(f"Sampled training rows to MAX_TRAIN_ROWS={MAX_TRAIN_ROWS}")

FEATURES = choose_features(train_df, test_df)
X = model_df[FEATURES]
y = model_df[TARGET_COL]
groups = model_df[GROUP_COL] if GROUP_COL in model_df.columns else None

numeric_features = X.select_dtypes(include=[np.number]).columns.tolist()
categorical_features = [col for col in FEATURES if col not in numeric_features]

print("Modeling rows:", len(model_df))
print("Feature count:", len(FEATURES))
print("Numeric features:", len(numeric_features))
print("Categorical features:", len(categorical_features))
print("First 40 features:", FEATURES[:40])
if groups is not None:
    print("Validation groups:", groups.nunique())
display(model_df[[TARGET_COL] + FEATURES[:10]].head())


## 4. Preprocessing and Models

Numerical features are median-imputed and scaled. Categorical features are imputed and one-hot encoded. The model list includes a linear baseline, tree ensemble baseline, and any available boosting libraries.

In [ ]:
# ============================================================
# 5. Preprocessing and model definitions
# ============================================================

def make_one_hot_encoder():
    try:
        return OneHotEncoder(handle_unknown="ignore", sparse_output=True)
    except TypeError:
        return OneHotEncoder(handle_unknown="ignore", sparse=True)


numeric_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ]
)

categorical_pipeline = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", make_one_hot_encoder()),
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_pipeline, numeric_features),
        ("cat", categorical_pipeline, categorical_features),
    ],
    remainder="drop",
)

models = {
    "ridge": Ridge(alpha=20.0),
    "extra_trees": ExtraTreesRegressor(
        n_estimators=200,
        min_samples_leaf=3,
        max_features=0.8,
        random_state=RANDOM_STATE,
        n_jobs=-1,
    ),
}

# RandomForest is included as an alternative tree model. ExtraTrees is usually faster here.
models["random_forest"] = RandomForestRegressor(
    n_estimators=150,
    min_samples_leaf=3,
    max_features=0.8,
    random_state=RANDOM_STATE,
    n_jobs=-1,
)

if lgb is not None:
    models["lightgbm"] = lgb.LGBMRegressor(
        objective="regression",
        n_estimators=900,
        learning_rate=0.035,
        num_leaves=63,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        random_state=RANDOM_STATE,
        n_jobs=-1,
        verbose=-1,
    )

if xgb is not None:
    models["xgboost"] = xgb.XGBRegressor(
        objective="reg:squarederror",
        eval_metric="rmse",
        n_estimators=700,
        learning_rate=0.04,
        max_depth=6,
        subsample=0.85,
        colsample_bytree=0.85,
        reg_lambda=2.0,
        tree_method="hist",
        random_state=RANDOM_STATE,
        n_jobs=-1,
    )

if catboost is not None:
    models["catboost"] = catboost.CatBoostRegressor(
        loss_function="RMSE",
        iterations=700,
        learning_rate=0.04,
        depth=6,
        random_seed=RANDOM_STATE,
        verbose=False,
        allow_writing_files=False,
    )

print("Models to train:", list(models.keys()))


## 5. Validation with Official Metric

The official metric is RMSE. For this wellbore competition, validation is grouped by well so rows from the same well never appear in both train and validation.

In [ ]:
# ============================================================
# 6. Validation
# ============================================================

def rmse(y_true, y_pred) -> float:
    try:
        return mean_squared_error(y_true, y_pred, squared=False)
    except TypeError:
        return mean_squared_error(y_true, y_pred) ** 0.5


def make_splits():
    if groups is not None and groups.nunique() >= 2:
        n_splits = min(N_SPLITS, groups.nunique())
        splitter = GroupKFold(n_splits=n_splits)
        return list(splitter.split(X, y, groups)), f"GroupKFold({n_splits}) by {GROUP_COL}"
    n_splits = min(N_SPLITS, len(X))
    splitter = KFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return list(splitter.split(X, y)), f"KFold({n_splits})"


splits, validation_name = make_splits()
print("Validation strategy:", validation_name)

results = []
oof_predictions = {}

for model_name, estimator in tqdm(models.items(), desc="Models"):
    fold_scores = []
    oof = np.full(len(X), np.nan)

    for fold, (train_idx, valid_idx) in enumerate(tqdm(splits, desc=model_name, leave=False), start=1):
        pipeline = Pipeline(
            steps=[
                ("preprocess", clone(preprocessor)),
                ("model", clone(estimator)),
            ]
        )
        pipeline.fit(X.iloc[train_idx], y.iloc[train_idx])
        valid_pred = pipeline.predict(X.iloc[valid_idx])
        fold_rmse = rmse(y.iloc[valid_idx], valid_pred)
        fold_scores.append(fold_rmse)
        oof[valid_idx] = valid_pred
        print(f"{model_name} | fold {fold} | RMSE: {fold_rmse:.5f}")

    oof_predictions[model_name] = oof
    results.append(
        {
            "model": model_name,
            "mean_rmse": float(np.mean(fold_scores)),
            "std_rmse": float(np.std(fold_scores)),
            "fold_scores": fold_scores,
        }
    )
    print(f"{model_name} | mean RMSE: {np.mean(fold_scores):.5f} +/- {np.std(fold_scores):.5f}")

results_df = pd.DataFrame(results).sort_values("mean_rmse").reset_index(drop=True)
print("\nValidation leaderboard, lower is better:")
display(results_df)

best_model_name = results_df.loc[0, "model"]
best_score = results_df.loc[0, "mean_rmse"]
print(f"Best validation model: {best_model_name} with mean RMSE {best_score:.5f}")

# Build out-of-fold ensembles from the strongest individual models.
# We only use validation predictions made on held-out folds, so this is leakage-safe.
ensemble_candidates = results_df["model"].head(min(ENSEMBLE_TOP_N, len(results_df))).tolist()
final_options = [
    {
        "strategy": "single_best",
        "model_names": [best_model_name],
        "weights": np.array([1.0]),
        "rmse": float(best_score),
    }
]

if len(ensemble_candidates) >= 2:
    oof_stack = np.column_stack([oof_predictions[name] for name in ensemble_candidates])
    valid_oof_rows = ~np.isnan(oof_stack).any(axis=1)

    simple_weights = np.ones(len(ensemble_candidates)) / len(ensemble_candidates)
    simple_oof = oof_stack[valid_oof_rows] @ simple_weights
    simple_rmse = rmse(y.iloc[valid_oof_rows], simple_oof)

    score_lookup = results_df.set_index("model")["mean_rmse"].to_dict()
    inverse_rmse_weights = np.array([1.0 / max(score_lookup[name], 1e-9) ** 2 for name in ensemble_candidates])
    inverse_rmse_weights = inverse_rmse_weights / inverse_rmse_weights.sum()
    weighted_oof = oof_stack[valid_oof_rows] @ inverse_rmse_weights
    weighted_rmse = rmse(y.iloc[valid_oof_rows], weighted_oof)

    final_options.extend(
        [
            {
                "strategy": "simple_average_ensemble",
                "model_names": ensemble_candidates,
                "weights": simple_weights,
                "rmse": float(simple_rmse),
            },
            {
                "strategy": "inverse_rmse_weighted_ensemble",
                "model_names": ensemble_candidates,
                "weights": inverse_rmse_weights,
                "rmse": float(weighted_rmse),
            },
        ]
    )

    print("\nEnsemble candidates:", ensemble_candidates)
    print(f"Simple average ensemble RMSE: {simple_rmse:.5f}")
    print(f"Weighted ensemble RMSE:      {weighted_rmse:.5f}")
    print("Weighted ensemble weights:")
    for name, weight in zip(ensemble_candidates, inverse_rmse_weights):
        print(f"  {name}: {weight:.4f}")
else:
    print("Not enough models for an ensemble; using the single best model.")

final_choice = min(final_options, key=lambda item: item["rmse"])
FINAL_STRATEGY = final_choice["strategy"]
FINAL_MODEL_NAMES = final_choice["model_names"]
FINAL_WEIGHTS = final_choice["weights"]
FINAL_VALIDATION_RMSE = final_choice["rmse"]

print("\nSelected final strategy:", FINAL_STRATEGY)
print("Selected models:", FINAL_MODEL_NAMES)
print("Selected validation RMSE:", f"{FINAL_VALIDATION_RMSE:.5f}")


## 6. Train Final Model or Ensemble and Create Submission

The selected final strategy is trained on all leakage-safe training rows. If an ensemble wins validation, each selected model is trained once and the predictions are averaged with validation-derived weights. The file `submission.csv` is written at the notebook working root, which is required for Kaggle code competitions.

In [ ]:
# ============================================================
# 7. Final training and submission
# ============================================================

final_pipelines = {}
test_prediction_stack = []

print(f"Training final strategy on all {len(X)} rows: {FINAL_STRATEGY}")
for model_name in tqdm(FINAL_MODEL_NAMES, desc="Final models"):
    pipeline = Pipeline(
        steps=[
            ("preprocess", clone(preprocessor)),
            ("model", clone(models[model_name])),
        ]
    )
    pipeline.fit(X, y)
    final_pipelines[model_name] = pipeline
    test_prediction_stack.append(pipeline.predict(test_df[FEATURES]))

test_prediction_stack = np.column_stack(test_prediction_stack)
test_predictions = test_prediction_stack @ FINAL_WEIGHTS

model_path = MODEL_DIR / f"final_{FINAL_STRATEGY}.joblib"
joblib.dump(
    {
        "models": final_pipelines,
        "features": FEATURES,
        "target": TARGET_COL,
        "validation": validation_name,
        "results": results_df,
        "final_strategy": FINAL_STRATEGY,
        "final_model_names": FINAL_MODEL_NAMES,
        "final_weights": FINAL_WEIGHTS,
        "final_validation_rmse": FINAL_VALIDATION_RMSE,
    },
    model_path,
)
print("Saved final model package:", model_path)
print("Final prediction weights:")
for model_name, weight in zip(FINAL_MODEL_NAMES, FINAL_WEIGHTS):
    print(f"  {model_name}: {weight:.4f}")

if SUBMISSION_ID_COL in test_df.columns:
    prediction_frame = pd.DataFrame(
        {
            SUBMISSION_ID_COL: test_df[SUBMISSION_ID_COL].values,
            SUBMISSION_TARGET_COL: test_predictions,
        }
    )
    submission = sample_submission[[SUBMISSION_ID_COL]].merge(prediction_frame, on=SUBMISSION_ID_COL, how="left")
elif len(test_predictions) == len(sample_submission):
    submission = sample_submission.copy()
    submission[SUBMISSION_TARGET_COL] = test_predictions
else:
    raise ValueError(
        f"Cannot align predictions: test rows={len(test_predictions)}, sample rows={len(sample_submission)}, "
        f"id column={SUBMISSION_ID_COL!r} not found in test data."
    )

missing_count = submission[SUBMISSION_TARGET_COL].isna().sum()
if missing_count:
    print(f"Warning: {missing_count} predictions were missing after alignment. Filling with train median.")
    submission[SUBMISSION_TARGET_COL] = submission[SUBMISSION_TARGET_COL].fillna(y.median())

submission = submission[sample_submission.columns]

# Required by Kaggle code competitions.
submission_path = OUTPUT_ROOT / "submission.csv"
submission.to_csv(submission_path, index=False)

# Local organizational copy.
local_submission_path = SUBMISSION_DIR / "submission.csv"
submission.to_csv(local_submission_path, index=False)

print("Saved Kaggle submission:", submission_path)
print("Saved local copy:", local_submission_path)
print("Submission shape:", submission.shape)
display(submission.head())

assert list(submission.columns) == list(sample_submission.columns)
assert len(submission) == len(sample_submission)
assert submission[SUBMISSION_TARGET_COL].notna().all()


## 7. Summary and Next Steps

After the notebook runs, read the printed validation leaderboard.

- **Best model:** the first row of `results_df`.
- **Features used:** all leakage-safe columns listed in `FEATURES`; raw IDs, `well`, target columns, and raw `TVT_input` are excluded.
- **Next things to try:** tune the best boosting model, add better GR/typewell correlation features, use fold-wise ensembling, and inspect validation errors by well/geology interval.